# Location Marginal Pricing - Initial GNN

## Colab setup (Colab only, skip on the Mac Mini)
First cell: every time the kernel starts, including after a restart (puts the repo on the import path, sets the DB and MLflow env vars, asks for the DB login). Second cell: once per Colab machine (joins the tailnet, forwards the DB (5432), MLflow (5000) and cache files (8765) on `angies-server` to `localhost` here, clones the branch, installs packages, downloads the cache; asks for the Tailscale auth key). Third cell: every time the kernel starts, after the files are on Colab. It makes `build_edges` use the fold's spread table built on the Mac Mini (`maintenance/spread_wide_cache.py`; downloaded the first time a fold is used, ~2-5 min) instead of pulling it from the DB, which takes ~3 hours over the Mac Mini's upload. Pick the fold with `FOLD` in the Setup cell. Colab secrets don't work from VS Code. See `references/lmp-pricing-model/decisions.md` (Colab GPU training).

In [ ]:
from getpass import getpass
import os
import sys

REPO = "/content/gridcast"
sys.path.insert(0, REPO)
os.environ.update({
    "TIMESCALEDB_HOST": "localhost", "TIMESCALEDB_PORT": "5432", "TIMESCALEDB_DB": "gridcast",
    "TIMESCALEDB_USER": input("DB user: "), "TIMESCALEDB_PASSWORD": getpass("DB password: "),
    "MLFLOW_TRACKING_URI": "http://localhost:5000", "MLFLOW_DISABLE_TELEMETRY": "true",
})

In [ ]:
import subprocess
import time

SERVER = "angies-server"

!curl -fsSL https://tailscale.com/install.sh | sh
!apt-get -qq install -y socat
subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--state=mem:"], stdout=open("/content/tailscaled.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(5)
subprocess.run(["tailscale", "up", f"--auth-key={getpass('Tailscale auth key: ')}", "--hostname=colab"], check=True)
for port in [5432, 5000, 8765]:
    subprocess.Popen(["socat", f"TCP-LISTEN:{port},bind=127.0.0.1,fork,reuseaddr", f"EXEC:tailscale nc {SERVER} {port}"], start_new_session=True)

!test -d {REPO} || git clone -q --branch lmp-pricing-model https://github.com/angieohaeri/gridcast.git {REPO}
!pip install -q mlflow-skinny==3.14.0 torch_geometric loguru python-dotenv psycopg2-binary holidays typer
!pip install -q --no-deps -e {REPO}

from lmp_model.dataset import CACHE_PATH

!mkdir -p {REPO}/data/processed/train_npz
!test -f {REPO}/data/processed/train_npz/window.npz || curl -fsSR -H "Host: {SERVER}:8765" -o {REPO}/data/processed/train_npz/window.npz http://localhost:8765/window.npz
!test -f {REPO}/data/processed/train_npz/exposure.npz || curl -fsSR -H "Host: {SERVER}:8765" -o {REPO}/data/processed/train_npz/exposure.npz http://localhost:8765/exposure.npz

In [ ]:
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import lmp_model.graph as G
from lmp_model.splits import walk_forward_folds


def cached_spread_wide(start, end):
    i = next(i for i, f in enumerate(walk_forward_folds()) if (f.train_start, f.val_start) == (start, end))
    path = Path(REPO) / "data" / "processed" / "train_npz" / f"spread_wide_fold{i}.npz"
    if not path.exists():
        subprocess.run(["curl", "-fsSR", "-H", "Host: angies-server:8765", "-o", str(path), f"http://localhost:8765/{path.name}"], check=True)
    w = np.load(path)
    return pd.DataFrame(w["values"], index=pd.to_datetime(w["times"], utc=True), columns=w["pnodes"])


G.load_spread_wide = cached_spread_wide

In [ ]:
import os
os.listdir(f"/{REPO}/data/processed/train_npz")

## Setup
Imports, load the cache, fold-0 split masks and scalers.

In [ ]:
import importlib
import lmp_model

# Run this line whenever you want to force-reload your code
importlib.reload(lmp_model)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

In [ ]:
from functools import lru_cache
import time

from loguru import logger
import mlflow
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch_geometric.nn import SAGEConv

from lmp_model.config import PROCESSED_DATA_DIR, log_datasets # Contents/ColabT4GPU/
from lmp_model.dataset import (
    CACHE_PATH,
    FEATURES,
    SYSTEM_FEATURES,
    ZONE_FEATURES,
    load_cache,
    load_nodes,
)
from lmp_model.features import fit_scalers, prepare
from lmp_model.graph import build_edges, edge_index
from lmp_model.splits import split_masks, walk_forward_folds


FOLD = 0
ENTRY = "E10"
folds = walk_forward_folds()
nodes = load_nodes()
pnode_ids = nodes.index
data = load_cache()
masks = split_masks(data["times"], folds[FOLD])
scalers = fit_scalers(data, masks["train"])
blocks = {name: prepare(data, mk, scalers) for name, mk in masks.items()}
{name: int(m.sum()) for name, m in masks.items()}

In [ ]:
# os.chdir(f"..")
os.listdir('content/gridcast/data/processed/train_npz')
os.m

In [ ]:
!cp gridcast/data/processed/train_npz/window.npz /content/Contents/ColabT4GPU/gridcast/

## Graph
Physical and correlation edges for fold 0, and hour batching.

In [ ]:
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HOURS_PER_BATCH = 16
Y_SCALE = 20.0

fold = folds[FOLD]
edges = build_edges(fold.train_start, fold.val_start)
full_ei = torch.from_numpy(edge_index(edges, pnode_ids)).long()
ei = full_ei
n_nodes = len(pnode_ids)


@lru_cache
def batch_edges(hours):
    return torch.cat([ei + i * n_nodes for i in range(hours)], dim=1).to(device)

## Feature sets
Which inputs each run uses, turned into tensors.

In [ ]:
def feature_names(feature_set):
    core4 = FEATURES[:4]
    history = FEATURES + SYSTEM_FEATURES
    return {"core4": core4, "history": history, "all": history + ["area_forecast"] + ZONE_FEATURES}[feature_set]


def node_inputs(block, feature_set):
    x = block["x"]
    if feature_set == "core4":
        return np.ascontiguousarray(x[..., :4])
    hours, nodes = x.shape[:2]
    parts = [x, np.broadcast_to(block["system"][:, None, :], (hours, nodes, block["system"].shape[1]))]
    if feature_set == "all":
        zone = data["node_zone"]
        zone_values = block["zone"][:, np.maximum(zone, 0)]
        zone_values[:, zone < 0] = 0
        parts += [block["area"][:, data["node_area"]][..., None], zone_values]
    return np.concatenate(parts, axis=-1)


def build_tensors(feature_set):
    return {
        name: (
            torch.from_numpy(node_inputs(b, feature_set)).to(device),
            torch.from_numpy(np.nan_to_num(b["y"]) / Y_SCALE).to(device),
            torch.from_numpy(b["loss_mask"]).to(device),
        )
        for name, b in blocks.items()
    }


def run_variant(feature_set_name, graph):
    global feature_set, tensors, ei
    feature_set = feature_set_name
    tensors = build_tensors(feature_set)
    ei = full_ei if graph == "full" else torch.empty(2, 0, dtype=torch.long)
    batch_edges.cache_clear()
    return train_gnn()

## Model and training loop
`SpreadGNN`, one epoch, prediction and scoring.

In [ ]:
class SpreadGNN(nn.Module):
    def __init__(self, n_nodes, n_in, hidden=64, node_dim=16, layers=2):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, node_dim)
        self.inp = nn.Linear(n_in + node_dim, hidden)
        self.convs = nn.ModuleList(SAGEConv(hidden, hidden) for _ in range(layers))
        self.out = nn.Linear(hidden, 1)

    def forward(self, x, edges):
        hours, nodes, _ = x.shape
        h = self.inp(torch.cat([x, self.node_emb.weight.expand(hours, -1, -1)], dim=-1)).relu()
        h = h.reshape(hours * nodes, -1)
        for conv in self.convs:
            h = h + conv(h, edges).relu()
        return self.out(h).reshape(hours, nodes)


def run_epoch(model, split, optimizer):
    x, y, mask = tensors[split]
    order = torch.randperm(len(x))
    n_batches = len(x) // HOURS_PER_BATCH
    total = 0.0
    for b in range(n_batches):
        idx = order[b * HOURS_PER_BATCH : (b + 1) * HOURS_PER_BATCH]
        pred = model(x[idx], batch_edges(len(idx)))
        m = mask[idx]
        loss = nn.functional.huber_loss(pred[m], y[idx][m], delta=1.0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
        if (b + 1) % 100 == 0:
            logger.info(f"  batch {b + 1}/{n_batches}, mean loss {total / (b + 1):.4f}")
    return total / n_batches


@torch.no_grad()
def predict(model, split):
    model.eval()
    x = tensors[split][0]
    out = [model(x[i : i + HOURS_PER_BATCH], batch_edges(len(x[i : i + HOURS_PER_BATCH]))) for i in range(0, len(x), HOURS_PER_BATCH)]
    model.train()
    return torch.cat(out).cpu().numpy() * Y_SCALE


def evaluate(pred, split):
    y = blocks[split]["y"]
    m = blocks[split]["loss_mask"]
    spike = m & (np.abs(y) >= np.quantile(np.abs(y[m]), 0.99))
    err = np.abs(pred - y)
    return {"mae": float(err[m].mean()), "spike_mae": float(err[spike].mean())}

## Training run
Hyperparameters, and `train_gnn` with MLflow logging (params, dataset samples, metrics).

In [ ]:
params = {
    "hidden": 64, "node_dim": 16, "layers": 2, "lr": 1e-3, "weight_decay": 1e-5,
    "hours_per_batch": HOURS_PER_BATCH, "max_epochs": 30, "patience": 5, "seed": 0,
    "huber_delta": Y_SCALE, "y_scale": Y_SCALE, "edge_variant": "all",
}
mlflow.set_experiment("gridcast-lmp-spread")


def train_gnn():
    graph = "full" if ei.shape[1] else "none"
    n_features = tensors["train"][0].shape[-1]
    edge_set = params["edge_variant"] if graph == "full" else "none"
    gpu = "_gpu" if device.type == "cuda" else ""
    run_name = f"gnn_{feature_set}_{edge_set}_f{FOLD}_s{params['seed']}{gpu}_{pd.Timestamp.now():%Y%m%d}"
    kinds = edges["kind"].value_counts().add_prefix("n_edges_").to_dict() if graph == "full" else {}
    torch.manual_seed(params["seed"])
    with mlflow.start_run(run_name=run_name) as run:
        out = PROCESSED_DATA_DIR / "runs" / run.info.run_id
        out.mkdir(parents=True)
        mlflow.set_tags({"entry": ENTRY, "fold": FOLD, "feature_set": feature_set, "graph": graph, "model": "sage_gnn", "device": device.type, "local_artifacts": str(out)})
        mlflow.log_params({
            **params, **kinds,
            "train_start": fold.train_start, "val_start": fold.val_start,
            "test_start": fold.test_start, "test_end": fold.test_end,
            "features": ",".join(feature_names(feature_set)), "n_features": n_features, "n_nodes": n_nodes,
            "n_edges_undirected": ei.shape[1] // 2,
            "clip_percentiles": "0.1,99.9", "nan_fill": 0,
        })
        log_datasets(
            {name: (tensors[name][0].cpu().numpy(), blocks[name]["y"], blocks[name]["loss_mask"], pd.DatetimeIndex(data["times"][masks[name]], tz="UTC")) for name in tensors},
            feature_names(feature_set), pnode_ids, CACHE_PATH, params["seed"],
        )
        logger.info(f"{run_name}: {len(tensors['train'][0])} train hours, {n_features} features, {ei.shape[1] // 2} edges")

        model = SpreadGNN(n_nodes, n_features, params["hidden"], params["node_dim"], params["layers"]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        best_val, best_epoch, stale = float("inf"), 0, 0
        for epoch in range(params["max_epochs"]):
            start = time.time()
            train_loss = run_epoch(model, "train", optimizer)
            val = evaluate(predict(model, "val"), "val")
            seconds = time.time() - start
            mlflow.log_metrics(
                {"train_loss": train_loss, "val_mae": val["mae"], "val_spike_mae": val["spike_mae"], "epoch_seconds": seconds},
                step=epoch,
            )
            logger.info(f"epoch {epoch}: train_loss {train_loss:.4f}, val_mae {val['mae']:.2f}, val_spike_mae {val['spike_mae']:.1f}, {seconds:.0f}s")
            if val["mae"] < best_val:
                best_val, best_epoch, stale = val["mae"], epoch, 0
                torch.save(model.state_dict(), out / "model.pt")
            else:
                stale += 1
                if stale >= params["patience"]:
                    logger.info(f"early stop: no improvement for {stale} epochs")
                    break

        model.load_state_dict(torch.load(out / "model.pt"))
        test = evaluate(predict(model, "test"), "test")
        zero = evaluate(np.zeros_like(blocks["test"]["y"]), "test")
        mlflow.log_metrics({
            "best_epoch": best_epoch, "best_val_mae": best_val,
            "test_mae": test["mae"], "test_spike_mae": test["spike_mae"],
            "zero_mae": zero["mae"], "zero_spike_mae": zero["spike_mae"],
            **{f"loss_mask_coverage_{name}": float(b["loss_mask"].mean()) for name, b in blocks.items()},
        })
        np.savez(out / "scalers.npz", **{f"{k}_{n}": a for k, v in scalers.items() for n, a in zip(["lo", "hi", "mean", "std"], v)})
        mlflow.log_artifacts(str(out))
        logger.info(f"best epoch {best_epoch}, val_mae {best_val:.2f}; test_mae {test['mae']:.2f}, test_spike_mae {test['spike_mae']:.1f} (zero {zero['mae']:.2f} / {zero['spike_mae']:.1f})")
    return test

## Run

In [ ]:
run_variant("history", "full")

## Feature-set experiments
Run A: the 15 Oct 27 inputs plus two target-day system features, copied into every node. Run B: system features fed once per hour as global context after message passing (`ContextGNN`).

In [ ]:
from lmp_model.dataset import query as db_query
import lmp_model.graph as G

LINE_SOURCES = ["line_single_outage", "line_only_candidate", "line_id_letters", "line_same_circuit", "facility_line"]
VIA_SOURCES = ["line_via_1", "line_via_2"]
VARIANTS = {"map_only": LINE_SOURCES + VIA_SOURCES, "direct_lines": VIA_SOURCES, "all_lines": []}
ABLATION_CSV = PROCESSED_DATA_DIR / "runs" / "edge_ablation_fold0.csv"

wide = G.load_spread_wide(fold.train_start, fold.val_start)
G.load_spread_wide = lambda start, end: wide


def edges_query(dropped):
    def q(sql, params=()):
        if "pnode_topology_edges" in sql:
            return db_query("SELECT sub_a, sub_b FROM raw_lmp.pnode_topology_edges WHERE NOT (source = ANY(%s::text[]));", (dropped,))
        return db_query(sql, params)
    return q


ablation = []
for variant, dropped in VARIANTS.items():
    G.query = edges_query(dropped)
    edges = G.build_edges(fold.train_start, fold.val_start)
    full_ei = torch.from_numpy(edge_index(edges, pnode_ids)).long()
    params["edge_variant"] = variant
    started = time.time()
    test = run_variant("history", "full")
    ablation.append({"variant": variant, **edges["kind"].value_counts().add_prefix("n_").to_dict(), **{f"test_{k}": v for k, v in test.items()}, "minutes": round((time.time() - started) / 60, 1)})
    pd.DataFrame(ablation).to_csv(ABLATION_CSV, index=False)
    logger.info(f"ablation {variant}: {ablation[-1]}")
pd.DataFrame(ablation)

In [ ]:
from lmp_model.dataset import CALENDAR_FEATURES, SYSTEM_TABLE_FEATURES
import lmp_model.graph as G

system_names = CALENDAR_FEATURES + SYSTEM_TABLE_FEATURES
STRONG = ["da_shadow_total", "da_constraint_count", "da_constraint_count_prev_day", "rt_constraint_count_prev_day", "rt_shadow_total_prev_day", "generation_mw_prev_day", "east_util", "bcpep_util", "bed_bla_util", "west_util"]
SETS = {"history_calendar": CALENDAR_FEATURES, "target_day2": CALENDAR_FEATURES + ["da_shadow_total", "da_constraint_count"], "context16": CALENDAR_FEATURES + STRONG}
DESCRIPTIONS = {
    "history_calendar": (
        "Baseline: the 15 inputs from Oct 27 (9 node features plus 6 calendar features), no system features.\n\n"
        "Rerun on the same hardware as runs A and B, because GPU and CPU runs differ slightly. "
        "A and B are compared against this run, not the earlier CPU result (29.28 test MAE)."
    ),
    "target_day2": (
        "Run A: a few system features, fed the old way.\n\n"
        "Inputs: the 15 from Oct 27 (9 node features plus 6 calendar features) plus just two system features, "
        "da_shadow_total and da_constraint_count. Those two tracked hourly spread size best (0.71 and 0.61). "
        "Like before, the two features are copied into every node's input.\n\n"
        "The question it answers: does any system feature help when fed this way, if we keep only the strongest two? "
        "It's compared against the 15-input run (29.28 test MAE on the original 1,505 nodes)."
    ),
    "context16": (
        "Run B: system features fed once, as global context.\n\n"
        "Each node's input is just its 9 node features. The 6 calendar features and 10 system features (the six strongest, "
        "plus four interface-utilization features) go in once per hour, through a small separate network. "
        "Its output is added to every node after message passing.\n\n"
        "The question it answers: if PJM-wide information sets the overall level for the hour without crowding the "
        "node-specific signal that neighbours exchange, do the system features stop hurting, or even help?"
    ),
}
_feature_names, _node_inputs, BaseGNN = feature_names, node_inputs, SpreadGNN


def feature_names(feature_set):
    return FEATURES + SETS[feature_set] if feature_set in SETS else _feature_names(feature_set)


def node_inputs(block, feature_set):
    if feature_set not in SETS:
        return _node_inputs(block, feature_set)
    cols = [system_names.index(f) for f in SETS[feature_set]]
    hours, n = block["x"].shape[:2]
    return np.concatenate([block["x"], np.broadcast_to(block["system"][:, None, cols], (hours, n, len(cols)))], axis=-1)


class ContextGNN(nn.Module):
    """Node features through message passing; system features once per hour, added after it."""

    def __init__(self, n_nodes, n_node_in, n_ctx, hidden=64, node_dim=16, layers=2):
        super().__init__()
        self.n_node_in = n_node_in
        self.node_emb = nn.Embedding(n_nodes, node_dim)
        self.inp = nn.Linear(n_node_in + node_dim, hidden)
        self.convs = nn.ModuleList(SAGEConv(hidden, hidden) for _ in range(layers))
        self.ctx = nn.Sequential(nn.Linear(n_ctx, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.out = nn.Linear(hidden, 1)

    def forward(self, x, edges):
        hours, nodes, _ = x.shape
        h = self.inp(torch.cat([x[..., : self.n_node_in], self.node_emb.weight.expand(hours, -1, -1)], dim=-1)).relu()
        h = h.reshape(hours * nodes, -1)
        for conv in self.convs:
            h = h + conv(h, edges).relu()
        h = h.reshape(hours, nodes, -1) + self.ctx(x[:, 0, self.n_node_in :])[:, None, :]
        return self.out(h).reshape(hours, nodes)


def context_gnn(n_nodes, n_in, hidden, node_dim, layers):
    return ContextGNN(n_nodes, len(FEATURES), n_in - len(FEATURES), hidden, node_dim, layers)


old_nodes = (nodes.reindex(pnode_ids).role != "constraint").to_numpy()


def scores(pred, cols):
    y, m = blocks["test"]["y"][:, cols], blocks["test"]["loss_mask"][:, cols]
    yy, err = y[m], np.abs(pred[:, cols] - y)[m]
    spike = np.abs(yy) >= np.quantile(np.abs(yy), 0.99)
    return round(float(err.mean()), 2), round(float(err[spike].mean()), 1)


def edges_query(dropped):
    def q(sql, params=()):
        if "pnode_topology_edges" in sql:
            return db_query("SELECT sub_a, sub_b FROM raw_lmp.pnode_topology_edges WHERE NOT (source = ANY(%s::text[]));", (dropped,))
        return db_query(sql, params)
    return q


G.query = edges_query(["hifld_osm"])
edges = G.build_edges(fold.train_start, fold.val_start)
full_ei = torch.from_numpy(edge_index(edges, pnode_ids)).long()
params["edge_variant"] = "all_lines"
round2 = []
for seed in [0, 1, 2]:
    params["seed"] = seed
    for feature_set, model_fn, label in [("history_calendar", BaseGNN, "Baseline: 15 inputs"), ("target_day2", BaseGNN, "A: 15 + 2 target-day, broadcast"), ("context16", context_gnn, "B: 9 node + 16 global context")]:
        SpreadGNN = model_fn
        params["model_variant"] = label
        run_variant(feature_set, "full")
        rid = mlflow.last_active_run().info.run_id
        mlflow.MlflowClient().set_tag(rid, "mlflow.note.content", DESCRIPTIONS[feature_set])
        model = model_fn(n_nodes, tensors["test"][0].shape[-1], params["hidden"], params["node_dim"], params["layers"]).to(device)
        model.load_state_dict(torch.load(PROCESSED_DATA_DIR / "runs" / rid / "model.pt"))
        pred = predict(model, "test")
        round2.append({"fold": FOLD, "seed": seed, "variant": label, "orig_1505": scores(pred, old_nodes), "all_1580": scores(pred, np.ones(len(old_nodes), bool)), "constraint_75": scores(pred, ~old_nodes), "best_val": round(mlflow.get_run(rid).data.metrics["best_val_mae"], 2), "run_id": rid})
        pd.DataFrame(round2).to_csv(PROCESSED_DATA_DIR / "runs" / f"feature_ablation_fold{FOLD}_round2.csv", index=False)
SpreadGNN = BaseGNN
params["seed"] = 0
print(pd.DataFrame(round2).to_string(index=False))